# Moteur de recommandation IoT — produits similaires & complementaires

Pour un produit de l'inventaire, ce notebook recommande **les produits similaires** (substituts du meme type) et **les produits complementaires** (a acheter avec).

## Lancer sur Google Colab
1. Mettre `inventory_export.csv` et `regles_recommandation.csv` a cote du notebook (ou les uploader).
2. **Runtime -> Run all** : le modele d'IA (embeddings) se telecharge automatiquement (gratuit).
3. Utiliser : `show_smart("ESP32")`, `show_smart("Batterie 18650")`, `evaluer()`.

## Cerveau LLM (optionnel mais recommande)
Un **LLM expert** choisit et **explique** les recommandations parmi les candidats du moteur (sans rien inventer).
Il s'active tout seul si une cle est presente : **HF_TOKEN** (deja la), ou mieux une cle gratuite **Groq**
(console.groq.com) / **Gemini** (aistudio.google.com) -- plus fluide que HF. Sans cle, le moteur marche seul.
Mets la cle dans les *Secrets* Colab (cle a gauche) ou dans .env.

## Comment ca marche
- Categorisation + extraction d'attributs (board, chimie, tension, capteur...).
- **SIMILAIRES** (`sim=`) : meme famille + meme attribut definissant ; pour les categories sans definissant (mecanique/connectique...), on exige un vrai lien de famille sinon rien. Score = vraie similarite (eleve = tres proche).
- **COMPLEMENTAIRES** (`compat=`) : compatibles d'une AUTRE famille. Le score est la COMPATIBILITE (board/connecteur/mot-cle) ; la tension seule ne suffit pas. Jamais un outil ni du bruit industriel.
- Embeddings `multilingual-e5-base` ; repli automatique TF-IDF si indisponible.

## Resultat mesure
Jeu de test etiquete (29 cas, matching MOT ENTIER pour ne pas se tromper soi-meme) : **precision ~93 %** (similaires ~89 %, complementaires ~98 %), **0 faute grave**. Lance `evaluer()` pour verifier.

## Regler le moteur
Editer `regles_recommandation.csv` (1 ligne/categorie : complement_categories, complement_mots_cles, exclure_mots_cles, attributs_similaire) puis re-executer a partir de l'etape Regles.


In [ ]:
!pip install pandas numpy scikit-learn sentence-transformers rapidfuzz openai anthropic -q
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import re, unicodedata, warnings
warnings.filterwarnings("ignore")
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
    print("OK Dependances -- mode semantique (sentence-transformers)")
except Exception:
    HAS_ST = False
    print("Repli automatique sur TF-IDF (sentence-transformers indisponible)")

## Etape 1 : Chargement des donnees

In [ ]:
candidate_files = [
    Path("inventory_export.csv"),
    Path("data/inventory_export.csv"),
    Path("/content/inventory_export.csv"),
    Path("/content/drive/MyDrive/inventory_export.csv"),
]
csv_path = next((p for p in candidate_files if p.exists()), None)
if csv_path is None:
    try:
        from google.colab import files
        print("Selectionne inventory_export.csv a uploader...")
        uploaded = files.upload()
        csv_path = Path(list(uploaded.keys())[0])
    except ModuleNotFoundError:
        raise FileNotFoundError("inventory_export.csv introuvable.")
df_raw = pd.read_csv(csv_path)
print(f"OK Charge depuis : {csv_path}")
print(f"OK {df_raw.shape[0]} lignes, {df_raw['Handle'].nunique()} produits uniques")
df_raw.head(3)

## Etape 2 : Nettoyage + Categorisation complete (objectif 0% 'autre')

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.lower().str.strip()

def normalize_text(text):
    """Texte filtre (sans mots-outils, sans lettres seules) -- pour categories + TF-IDF."""
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
    text = re.sub(r"[^a-z0-9\s]"," ",text)
    stop = {"de","des","du","la","le","les","un","une","et","au","aux","pour","avec","sur","dans","par","ou","en"}
    return " ".join(t for t in text.split() if len(t)>1 and t not in stop)

def norm_raw(text):
    """Texte brut (GARDE les lettres seules : 'type c', 'pile d') -- pour l'extraction d'attributs."""
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+"," ",text).strip()

# ============================================================================
# BRIQUE 0 -- CATEGORISATION COMPLETE (objectif : 0 % de "autre")
# ============================================================================
CATEGORY_RULES = {
 "capteur":     ["capteur","capteurs","senseur","sensor","dht11","dht22","ds18b20","hc-sr04","hcsr04","pir","ultrason","mpu6050","mpu9250","bmp","bme280","mq2","mq3","mq135","ldr","acs712","sct013","gy-","lm35","tof","photoresistance","accelerometre","gyroscope","camera","debimetre","yf-s201","flow","ad8232","ecg","hc-sr501","encodeur","mlx","reed","hx711","load cell","fin de course"],
 "carte":       ["raspberry","rpi","arduino","esp32","esp8266","nodemcu","wemos","microbit","stm32","atmega","jetson","wroom","attiny","carte developpement","mega2560","leonardo"],
 "mobilite":    ["voiture","telecommandee","telecommande","drone","quadcopter","helices","helice","roue","roues","chassis","4wd","2wd","robot","bateau","avion","mecanum","suiveur"],
 "moteur":      ["moteur","motor","servo","servomoteur","stepper","nema17","nema23","28byj","pompe","ventilateur","brushless","pas a pas","mg90","mg90s","mg995","mg996","sg90","ds3218","actionneur","vibreur"],
 "led":         ["led","ruban","bande","ampoule","lampe","lumiere","spectre","rgb","neon","matrice","ws2812","cob","horticole","projecteur","spot","luminaire","strip"],
 "batterie":    ["pile","piles","batterie","battery","18650","21700","26650","14500","lipo","li-po","lithium","accu","alcaline","alkaline","nimh","ni-mh","vape","cr2032","cr2016","cr1620","r20","r14"],
 "chargeur":    ["chargeur","charging","chargement","tp4056","imax"],
 "alimentation":["alimentation","tension","regulateur","regulator","convertisseur","transformateur","decoupage","buck","boost","abaisseur","elevateur","pwm","7805","79l12","onduleur","step-down","step-up","fusible"],
 "rf":          ["433mhz","rf","emetteur","recepteur","bluetooth","ble","wifi","gsm","gprs","gps","nrf24","lora","zigbee","sim800","sim900","antenne","hm-10","nfc","rfid"],
 "electronique":["module","circuit","relais","relay","lcd","oled","hdmi","i2c","spi","74hc","mosfet","registre","potentiometre","potentiometer","ecran","interrupteur","bouton","contacteur","rtc","driver","l298","shield","bouclier","pcb","prototype","perforee","controleur","cnc","gravure","expansion","stc-1000","uln2003","afficheur","segments","tm1637","max7219","clavier","membrane","ft232","rs232","cp2102","ch340","keypad","joystick","dip switch","horloge","ne555"],
 "composant":   ["diode","diodes","resistance","resistances","condensateur","transistor","zener","quartz","cristal","inductance","1n4148","1n4007","thyristor","triac","optocoupleur","triode","optotriac"],
 "connectique": ["cable","cables","fil","fils","connecteur","connecteurs","cosses","cosse","jumper","cavalier","dupont","barette","barrette","prise","borne","bornier","header","nappe","gaine","thermoretractable","domino","carte memoire","micro sd","microsd","sd card","cordon","rallonge","serre cable"],
 "mesure":      ["multimetre","testeur","oscilloscope","amperemetrique","pince ampere","compteur","balance","luxmetre","thermometre","wattmetre","pzem","ph metre","electrode","sonde","frequencemetre","generateur de signal"],
 "soudure":     ["souder","soudure","etain","flux","panne","dessoudage","fer a souder","desoudage"],
 "solaire":     ["solaire","photovoltaique","photovolta","mppt","panneau solaire"],
 "audio":       ["haut-parleur","haut parleur","microphone","amplificateur","buzzer","speaker","ampli","sonore","hp","mp3","jack audio"],
 "outillage":   ["tournevis","cle","cles","clef","pince","pinces","scie","cutter","lame","lames","foret","forets","marteau","coffret","outil","outils","douille","douilles","embout","embouts","percage","brucelle","mandrin","cliquet","perceuse","visseuse","meuleuse","burin","lime","etau","cle mixte","jeu de","pistolet","colle","peinture","disque","brosse","meule","nettoyeur","levage","electrogene","laser","niveau","compresseur","ponceuse","rabot","agrafeuse","riveteuse","decapeur","imprimante","creality","filament","truelle","spatule","pinceau","rouleau","echelle","escabeau","cric","palan","aspirateur","disqueuse","sangle"],
 "mecanique":   ["vis","ecrou","ecrous","boulon","rondelle","rondelles","rivet","roulement","engrenage","courroie","poulie","ressort","rail","profile","entretoise","boitier","coque","dissipateur","heatsink","radiateur","colonnette","equerre","charniere","glissiere","tige filetee"],
 "electrique":  ["disjoncteur","contacteur","sectionneur","chint","parafoudre","goulotte","armoire"],
}
ACCESSORY = {"lcd","oled","ecran","afficheur","ruban","boitier","coque","dissipateur",
             "ventilateur","camera","shield","transformateur","adaptateur","alimentation","chargeur"}
CAT_ORDER = ["chargeur","batterie","led","alimentation","moteur","rf","electronique","composant",
             "connectique","mesure","soudure","solaire","audio","outillage","mecanique","electrique"]

TOOL_BRANDS = {"total","wadfow","harden","yato","tolsen","ingco","stanley","bosch","makita",
               "dewalt","milwaukee","creality","deli"}
def is_tool_brand(title):
    return any(b in norm_raw(title).split() for b in TOOL_BRANDS)

def _has(n, words, kws):
    for k in kws:
        if " " in k:
            if k in n: return True
        elif len(k) <= 3:
            if k in words: return True
        else:
            if any(k in w for w in words) or k in n: return True
    return False

def infer_category(title):
    n = normalize_text(title); words = set(n.split()); wr = set(norm_raw(title).split())
    if _has(n, words, CATEGORY_RULES["mobilite"]): return "mobilite"
    if _has(n, words, CATEGORY_RULES["capteur"]):  return "capteur"
    if _has(n, words, CATEGORY_RULES["carte"]) and not (wr & ACCESSORY): return "carte"
    for cat in CAT_ORDER:
        if _has(n, words, CATEGORY_RULES[cat]): return cat
    if is_tool_brand(title): return "outillage"
    return "autre"

# ============================================================================
# SPECS
# ============================================================================
SPEC_PATTERNS = {
 "capacity_mah":(r'\b(\d+(?:[.,]\d+)?)\s*mah\b',50,100000),
 "capacity_ah": (r'\b(\d+(?:[.,]\d+)?)\s*ah\b',0.2,500),
 "voltage_v":   (r'\b(\d+(?:[.,]\d+)?)\s*v\b',0.5,600),
 "power_w":     (r'\b(\d+(?:[.,]\d+)?)\s*w\b',0.1,5000),
 "count":       (r'\b(\d+)\s*(?:pcs|pieces|broches|pin|leds?|canaux|ch)\b',1,10000),
}
def extract_specs(title):
    t = unicodedata.normalize("NFKD", str(title)).encode("ascii","ignore").decode("ascii").lower().replace("*"," ")
    out={}
    for name,(pat,lo,hi) in SPEC_PATTERNS.items():
        v=[float(x.replace(",",".")) for x in re.findall(pat,t)]; v=[x for x in v if lo<=x<=hi]
        if v: out[name]=max(v)
    return out
UPGRADE_SPECS=["capacity_mah","capacity_ah","power_w","count"]
def primary_spec(specs):
    for k in UPGRADE_SPECS:
        if k in specs: return k,specs[k]
    return None,0.0

# ============================================================================
# BRIQUE 1 -- ATTRIBUTS
# ============================================================================
LIION_FF=["18650","21700","26650","14500","16340","18500","20700"]; COIN_FF=["cr2032","cr2016","cr1620","cr2025","cr2450","lr44","ag13"]
def a_form_factor(n):
    for f in LIION_FF:
        if f in n: return f
    for f in COIN_FF:
        if f in n: return "coin"
    if re.search(r'\b(rl20|pile d)\b',n): return "d"
    if re.search(r'\b(rl14|pile c)\b',n): return "c"
    if re.search(r'\b(aaa|lr03|r03)\b',n): return "aaa"
    if re.search(r'\b(aa|lr6|r6)\b',n): return "aa"
    if re.search(r'\b9v\b',n): return "9v"
    if re.search(r'\bnema\s?17\b',n): return "nema17"
    if re.search(r'\bnema\s?23\b',n): return "nema23"
    if "5mm" in n: return "5mm"
    if "3mm" in n: return "3mm"
    for c in ["5050","2835","3528"]:
        if c in n: return c
    return None
def a_chemistry(n,ff):
    if "lipo" in n or "li-po" in n or "polymer" in n: return "lipo"
    if ff in LIION_FF or "li-ion" in n or "liion" in n or "lithium" in n: return "lithium"
    if "nimh" in n or "ni-mh" in n: return "nimh"
    if "alcaline" in n or "alkaline" in n or ff in ("d","c","aa","aaa","9v"): return "alkaline"
    if "plomb" in n or "agm" in n or "ultracell" in n or "acide" in n: return "lead"
    if "vape" in n or "pod" in n: return "vape"
    if ff=="coin": return "lithium"
    return None
def a_connector(n):
    if "type-c" in n or "type c" in n or "usb-c" in n or "usb c" in n: return "usb_c"
    if "micro usb" in n or "micro-usb" in n or "microusb" in n: return "usb_micro"
    if "type-b" in n or "type b" in n or "usb-b" in n: return "usb_b"
    if "xt60" in n: return "xt60"
    if "jst" in n: return "jst"
    if "hdmi" in n: return "hdmi"
    if "rj45" in n or "ethernet" in n: return "rj45"
    if "jack" in n: return "jack"
    if "usb" in n: return "usb_a"
    return None
def a_connectivity(n):
    if "wifi" in n or "wi-fi" in n: return "sans" if ("sans wifi" in n or "without wifi" in n) else "wifi"
    if "bluetooth" in n or re.search(r'\bble\b',n): return "bluetooth"
    if "433" in n or "nrf24" in n or "lora" in n or "zigbee" in n or re.search(r'\brf\b',n): return "rf"
    if "gsm" in n or "gprs" in n: return "gsm"
    if "gps" in n: return "gps"
    if "sans fil" in n: return "wireless"
    return None
def a_control(n):
    if "application" in n or re.search(r'\bappli\b',n) or re.search(r'\bapp\b',n): return "app"
    if "wifi" in n: return "wifi"
    if "bluetooth" in n: return "bluetooth"
    if "telecommand" in n or "radiocommand" in n or re.search(r'\brc\b',n): return "rc"
    return None
def a_vehicle(n):
    w=set(n.split())
    if "voiture" in n or "automobile" in n: return "voiture"
    if "drone" in n or "quadcopter" in n: return "drone"
    if "bateau" in n: return "bateau"
    if "avion" in n: return "avion"
    if "moto" in w: return "moto"
    if "tank" in w: return "char"
    if "robot" in n: return "robot"
    return None
def a_board(n):
    w=set(n.split())
    if "esp32" in n: return "esp32"
    if "esp8266" in n or "nodemcu" in n or "wemos" in n or "esp-12" in n: return "esp8266"
    if "raspberry" in n or "rpi" in w or "pi" in w: return "raspberry"
    if "arduino" in n: return "arduino"
    if w & {"uno","nano","leonardo","mega"}: return "arduino"
    if "microbit" in n: return "microbit"
    if "stm32" in n: return "stm32"
    if "jetson" in n: return "jetson"
    return None
def a_led_form(n):
    if "ruban" in n or "bande" in n or "strip" in n: return "strip"
    if "ampoule" in n or "spot" in n or re.search(r'\be14\b',n) or re.search(r'\be27\b',n) or "gu10" in n: return "bulb"
    if "matrice" in n or "ws2812" in n or "8x8" in n or "neopixel" in n: return "matrix"
    if "cob" in n or "horticole" in n or "spectre" in n: return "cob"
    if "infrarouge" in n or "850nm" in n or "940nm" in n or re.search(r'\bir\b',n): return "ir"
    if "afficheur" in n or "7 segment" in n or "segments" in n: return "display"
    if "neon" in n: return "neon"
    if "5mm" in n or "3mm" in n: return "discrete"
    if "lampe" in n or "projecteur" in n or "luminaire" in n: return "bulb"
    return None
def a_voltage_bucket(specs):
    v=specs.get("voltage_v")
    if v is None: return None
    for b in (3.3,5,12,24,220):
        if abs(v-b)<=max(0.6,b*0.12): return b
    return round(v)
def a_psu(n):
    if "buck" in n or "abaisseur" in n or "step down" in n: return "buck"
    if "boost" in n or "elevateur" in n or "step up" in n: return "boost"
    if "decoupage" in n or "transformateur" in n or "ac dc" in n: return "acdc"
    if "pwm" in n: return "pwm"
    return None
def a_rftech(n):
    if "433" in n: return "433"
    if "bluetooth" in n or re.search(r'\bble\b',n) or "hm-10" in n: return "bluetooth"
    if "wifi" in n: return "wifi"
    if "gsm" in n or "gprs" in n or "sim800" in n: return "gsm"
    if "gps" in n: return "gps"
    if "lora" in n: return "lora"
    if "zigbee" in n: return "zigbee"
    if "nrf24" in n: return "nrf24"
    if "nfc" in n or "rfid" in n: return "nfc"
    return None
def a_component(n):
    for c in ["resistance","condensateur","transistor","zener","diode","quartz","inductance","optocoupleur","thyristor","triac"]:
        if c in n: return c
    return None
def a_module_fn(n):
    # reconnait la FONCTION d'un circuit / module (registre, timer, rtc, driver, ampli...)
    if "relais" in n or "relay" in n: return "relais"
    if "registre" in n or "decalage" in n or "shift register" in n or "74hc595" in n or "74hc164" in n or "74hc165" in n: return "registre"
    if "rtc" in n or "horloge" in n or "ds1302" in n or "ds3231" in n: return "rtc"
    if "ne555" in n or "timer" in n or re.search(r'\b555\b',n): return "timer"
    # display = un AFFICHEUR, PAS un objet qui contient un ecran (stylo 3D, jauge LCD, fer a ecran...)
    if ("lcd" in n or "oled" in n or "afficheur" in n or "tft" in n or "nextion" in n
            or "1602" in n or "2004" in n or "12864" in n or "7 segment" in n or "segments" in n
            or re.search(r'\becran\b',n)):
        if not re.search(r'\b(stylo|jauge|fer a souder|imprimante|niveau|tournevis|3d|socle|angle|'
                         r'oscilloscope|microscope|loupe|camera|televiseur|moniteur|balance|'
                         r'thermometre|multimetre|testeur|pince|perceuse|visseuse)\b', n):
            return "display"
    if "l298" in n or "uln2003" in n or "driver" in n or "darlington" in n: return "driver"
    if "amplificateur" in n or "opamp" in n or "lm358" in n or "lm741" in n or "tda" in n: return "ampli"
    if "potentiometre" in n or "potentiometer" in n: return "potentiometre"
    if "optocoupleur" in n: return "opto"
    if "convertisseur" in n: return "convertisseur"
    if "interrupteur" in n: return "interrupteur"
    if "bouton" in n: return "bouton"
    if re.search(r'\b74(hc|hct|ls)\d',n): return "logique"
    return None
def a_motor(n):
    if "servo" in n or re.search(r'\b(mg90|mg90s|mg995|mg996|sg90|ds3218)\b',n): return "servo"
    if "stepper" in n or "nema" in n or "28byj" in n or "pas a pas" in n: return "stepper"
    if "pompe" in n: return "pompe"
    if "ventilateur" in n: return "ventilateur"
    if "moteur" in n or "motor" in n: return "dc"
    return None
def a_sensor(n):
    if "camera" in n: return "camera"
    if "dht" in n or "ds18b20" in n or "lm35" in n or "temperature" in n or "thermom" in n: return "temperature"
    if "hc-sr04" in n or "ultrason" in n or "sharp" in n or "tof" in n or "distance" in n: return "distance"
    if "pir" in n or "mouvement" in n or "motion" in n or "hc-sr501" in n: return "motion"
    if "mq" in n or "gaz" in n or "co2" in n or "pollution" in n: return "gaz"
    if "bmp" in n or "bme" in n or "pression" in n: return "pression"
    if "acs712" in n or "sct" in n or "courant" in n: return "courant"
    if "mpu" in n or "gyro" in n or "accel" in n: return "imu"
    if "ldr" in n or "lux" in n: return "lumiere"
    if "humidite" in n or "pluie" in n: return "humidite"
    return None

def extract_attrs(title, specs):
    n = norm_raw(title); ff = a_form_factor(n)
    return {"form_factor":ff,"chemistry":a_chemistry(n,ff),"connector":a_connector(n),
            "connectivity":a_connectivity(n),"control":a_control(n),"vehicle":a_vehicle(n),
            "board":a_board(n),"led_form":a_led_form(n),"voltage_bucket":a_voltage_bucket(specs),
            "psu":a_psu(n),"rftech":a_rftech(n),"component":a_component(n),
            "module_fn":a_module_fn(n),"motor":a_motor(n),"sensor":a_sensor(n)}

# ============================================================================
# Construction des colonnes
# ============================================================================
df["product_handle"] = df["handle"].astype(str).str.strip()
df["product_title"]  = df["title"].astype(str).str.strip()
df["sku"]            = df["sku"].astype(str).str.strip() if "sku" in df.columns else ""

for col in ["available (not editable)","on hand (current)","on hand (new)"]:
    if col in df.columns:
        df[col] = df[col].replace("not stocked",0)
        df[col] = pd.to_numeric(df[col],errors="coerce").fillna(0).astype(int)

df["clean_title"] = df["product_title"].apply(normalize_text)
df["category"]    = df["product_title"].apply(infer_category)

# Repli INTELLIGENT : tout produit encore "autre" herite de la categorie de son plus proche voisin
_mask = df["category"] == "autre"
if _mask.any():
    _known = df.loc[~_mask]
    _v  = TfidfVectorizer(max_features=3000).fit(df["clean_title"])
    _nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(_v.transform(_known["clean_title"]))
    _, _ind = _nn.kneighbors(_v.transform(df.loc[_mask, "clean_title"]))
    df.loc[_mask, "category"] = _known["category"].to_numpy()[_ind[:, 0]]

df["specs"]   = df["product_title"].apply(extract_specs)
df["attrs"]   = df.apply(lambda r: extract_attrs(r["product_title"], r["specs"]), axis=1)
df["is_tool"] = df["product_title"].apply(is_tool_brand)

uniq = df.drop_duplicates("product_handle")
print("OK Categorisation complete + attributs extraits")
dist = uniq["category"].value_counts()
print("\nDistribution des categories :")
print(dist.to_string())
print(f"\n   'autre' = {100*dist.get('autre',0)/len(uniq):.1f}%   "
      f"| produits avec >=1 attribut = {100*(uniq['attrs'].apply(lambda a: any(v is not None for v in a.values()))).mean():.0f}%")

In [ ]:
# === Affinage des categories (matching sur titre NORMALISE : sans accents/ponctuation) ===
_tn = df["product_title"].map(norm_raw)

# ============================================================================
# 0) FAUTE "POUR <board>" : un ACCESSOIRE pour Arduino/ESP/RPi mal range en 'carte'
#    (ex: "jeux de fils ... pour arduino") -> remis dans sa VRAIE famille.
#    On ne touche PAS aux vraies cartes (uno/nano/mega/wroom/dev kit) ni au "+ cable" final.
# ============================================================================
_pour = _tn.str.contains(r"pour (arduino|esp32|esp8266|raspberry|rpi|micro ?bit|stm32|nodemcu)|compatible (arduino|raspberry)", na=False)
_board_noun = _tn.str.contains(r"\buno\b|\bnano\b|\bmega\b|wroom|wrover|dev.?kit|carte de developpement|leonardo|\bpico\b|atmega|esp32 s3|esp32 c3|esp 12|jetson", na=False)
_acc = (df["category"]=="carte") & _pour & (~_board_noun)
df.loc[_acc, "category"] = "connectique"                                                                      # defaut : cables / fils
df.loc[(df["category"]=="connectique") & _pour & _tn.str.contains(r"\becran\b|afficheur|\blcd\b|oled|matrice|shield|bouclier|module", na=False), "category"] = "electronique"
df.loc[(df["category"]=="connectique") & _pour & _tn.str.contains(r"boitier|coque|housse|dissipateur|\bsupport\b|ventilateur|radiateur", na=False), "category"] = "mecanique"
df.loc[(df["category"]=="connectique") & _pour & _tn.str.contains(r"\bchargeur\b|\balimentation\b|adaptateur secteur|transformateur", na=False), "category"] = "alimentation"
# retirer le faux board de ces accessoires (sinon ils matchent les cartes a tort)
for _i in df.index[_acc]:
    _a = dict(df.at[_i, "attrs"]); _a["board"] = None; df.at[_i, "attrs"] = _a

# 1) AUDIO (enceintes, transmetteur FM, recepteur audio) -> 'audio'
_audio = _tn.str.contains(r"haut parleur|enceinte|speaker|soundtech|barre de son|"
                          r"transmetteur fm|transmetteur audio|adaptateur audio|recepteur audio|mains libre", na=False)
df.loc[_audio, "category"] = "audio"

# 2) INDUSTRIEL (automates, PLC, disjoncteurs...) -> 'electrique' (aucun complement IoT)
_indus = _tn.str.contains(r"automate|simatic|s7 1200|s7 200|s7 300|\bplc\b|industrial|industruino|"
                          r"disjoncteur|contacteur|sectionneur|variateur|profibus|\bvfd\b|triphas", na=False)
df.loc[_indus, "category"] = "electrique"

# 3) OUTILS (moteur auto + electroportatif) dans 'moteur' -> 'outillage'
_tool = _tn.str.contains(r"compressiometre|compression moteur|soupape|depose|calage|distribution|vilebrequin|"
                         r"culasse|injecteur|support moteur|cale moteur|extracteur|arrache|demonte|"
                         r"cle a choc|visseuse|perceuse|meuleuse|disqueuse|cisaille|souffleur|ponceuse|rabot|"
                         r"\bscie\b|testeur.*moteur", na=False)
df.loc[_tool & (df["category"]=="moteur"), "category"] = "outillage"

# 4) Cartes de developpement (camera incluse) -> 'carte'
_devboard = (_tn.str.contains(r"carte de developpement|carte developpement|dev board|dev kit|wroom|wrover|nodemcu|esp32 cam|esp cam", na=False)
             & _tn.str.contains(r"esp32|esp8266|esp 12|arduino|raspberry|stm32|rp2040|pico|microbit", na=False))
df.loc[_devboard, "category"] = "carte"

# 4b) Une carte ESP32/ESP8266 SANS fonction de module = une CARTE
_espcard = df["attrs"].apply(lambda a: a.get("board") in ("esp32","esp8266","rp2040") and a.get("module_fn") is None) \
           & df["category"].isin(["capteur","rf","electronique"])
df.loc[_espcard, "category"] = "carte"

# 5) Accessoires de BATTERIE -> vraie categorie
df.loc[_tn.str.contains(r"\btesteur\b", na=False) & (df["category"]=="batterie"), "category"] = "mesure"
df.loc[_tn.str.contains(r"\bbms\b|carte de protection|protection batterie", na=False) & (df["category"]=="batterie"), "category"] = "composant"
df.loc[_tn.str.contains(r"boitier|power bank|porte pile", na=False) & (df["category"]=="batterie"), "category"] = "mecanique"

# 6) FPGA / Altera ne sont PAS des Arduino -> corriger le faux attribut board
_fpga = _tn.str.contains(r"fpga|altera|cyclone|spartan|xilinx|lattice", na=False)
for _i in df.index[_fpga]:
    _a = dict(df.at[_i, "attrs"]); _a["board"] = "fpga"; df.at[_i, "attrs"] = _a

# 7) Panneau solaire mal classe : le mot "panneau" CONTIENT "panne" (mot-cle soudure) -> faux match.
_solpan = _tn.str.contains(r"panneau solaire|panneau photovolta|cellule solaire|mono ?cristallin|polycristallin|module solaire", na=False)
df.loc[_solpan, "category"] = "solaire"

# 8) Condensateurs CBB61/CBB65 (ventilateur/demarrage/climatiseur) mal classes 'moteur' -> composant (secteur 220/450V).
_cbb = _tn.str.contains(r"\bcbb6\d\b|condensateur.*(ventilateur|demarrage|moteur|climatiseur)", na=False) & (df["category"]=="moteur")
df.loc[_cbb, "category"] = "composant"

# 9) Faux 'moteur' : rapporteur d'angle (mesure), pompe nettoyant/tuyau/boite vide (outillage/mecanique).
df.loc[_tn.str.contains(r"rapporteur", na=False) & (df["category"]=="moteur"), "category"] = "mesure"
df.loc[_tn.str.contains(r"nettoyant|tuyau|boite vide|separation d ecran", na=False) & (df["category"]=="moteur"), "category"] = "outillage"

_nb_acc = int(_acc.sum())
print(f"OK Affinage : {_nb_acc} accessoires 'pour <board>' reclasses + audio/industriel/outils/ESP/batterie/FPGA + solaire/CBB")

## Etape 3 : Regles de recommandation (fichier CSV editable)
Le fichier `regles_recommandation.csv` pilote la logique : criteres du 'meme type' + complements. Edite-le sans toucher au code.

In [ ]:
# ============================================================================
# REGLES DE RECOMMANDATION (source = ce notebook ; exportees en CSV editable)
#   colonnes : attributs_similaire | complement_categories |
#              complement_categories_toutes | complement_mots_cles |
#              exclure_mots_cles  (NOUVEAU : blackliste de complements)
# IMPORTANT : REGEN_RULES=True reecrit toujours le CSV depuis _DEFAULT_RULES.
#   -> Pour personnaliser : edite _DEFAULT_RULES ci-dessous, puis re-execute a partir de l'Etape 5.
# ============================================================================
REGEN_RULES = True
RULES_CSV = Path("regles_recommandation.csv")

_DEFAULT_RULES = [
 {"categorie":"batterie","attributs_similaire":"form_factor;chemistry","complement_categories":"chargeur;composant;mecanique;connectique","complement_categories_toutes":"chargeur","complement_mots_cles":"bms;protection;support;porte pile;boitier;holder;chargeur;18650;module charge","exclure_mots_cles":"diode;triac;mosfet;transistor;resistance;condensateur;foret;projecteur;capteur;relais;moteur;ruban led","exemple":"18650 -> chargeur 18650 / BMS / support (jamais diode/TRIAC)"},
 {"categorie":"chargeur","attributs_similaire":"chemistry;connector","complement_categories":"batterie;connectique","complement_categories_toutes":"batterie","complement_mots_cles":"batterie;pile;accu;18650;lipo;li-ion;cellule","exclure_mots_cles":"foret;vis;peinture;moteur","exemple":"Chargeur 18650 -> cellules 18650"},
 {"categorie":"led","attributs_similaire":"led_form;voltage_bucket","complement_categories":"alimentation;electronique;connectique","complement_categories_toutes":"","complement_mots_cles":"alimentation;transformateur;controleur;dimmer;connecteur ruban;profile;amplificateur rgb;bande;ruban","exclure_mots_cles":"foret;vis;peinture;moteur;capteur;fpga;altera;cyclone;diode;projecteur","exemple":"Ruban LED 12V -> alim 12V / controleur RGB 12V"},
 {"categorie":"alimentation","attributs_similaire":"voltage_bucket;connector","complement_categories":"carte;connectique;led;electronique;mecanique","complement_categories_toutes":"","complement_mots_cles":"raspberry;arduino;esp32;micro sd;carte memoire;boitier;dissipateur;ruban led;ruban;jack","exclure_mots_cles":"foret;vis;peinture;cle;douille;fpga;altera;cyclone;diode;triac;projecteur;domino","exemple":"Alim 5V Raspberry -> Raspberry / carte SD / boitier / dissipateur"},
 {"categorie":"carte","attributs_similaire":"board;connectivity","complement_categories":"capteur;connectique;alimentation;led;electronique;composant","complement_categories_toutes":"capteur","complement_mots_cles":"breadboard;plaque essai;dupont;jumper;cable usb;micro sd;carte memoire;ecran;oled;lcd;afficheur;capteur;alimentation;boitier;dissipateur;relais","exclure_mots_cles":"foret;perceuse;vis;cle;douille;peinture;disjoncteur;fpga;altera;cyclone;visseuse;clé à chocs","exemple":"Arduino/ESP -> capteurs / dupont / alim / ecran"},
 {"categorie":"capteur","attributs_similaire":"sensor","complement_categories":"carte;connectique;electronique","complement_categories_toutes":"carte","complement_mots_cles":"arduino;esp32;raspberry;dupont;jumper;breadboard;resistance;ecran;lcd;oled;module;relais","exclure_mots_cles":"foret;perceuse;vis;cle;peinture;fpga;altera;cyclone;clé à chocs;projecteur","exemple":"Capteur DHT11/PIR -> carte MCU / dupont / ecran"},
 {"categorie":"moteur","attributs_similaire":"motor;voltage_bucket","complement_categories":"carte;electronique;alimentation;batterie;mecanique","complement_categories_toutes":"","complement_mots_cles":"driver;l298;a4988;controleur;esc;arduino;esp32;roue;helice;batterie;lipo;alimentation;servo;pont en h","exclure_mots_cles":"foret;perceuse;vis;peinture;clé à chocs;cle a choc;visseuse;disqueuse;meuleuse;fpga","exemple":"Moteur -> driver L298/ESC / carte / batterie"},
 {"categorie":"mobilite","attributs_similaire":"vehicle;control","complement_categories":"batterie;moteur;capteur;electronique;connectique;chargeur;rf","complement_categories_toutes":"moteur","complement_mots_cles":"telecommande;radiocommande;roue;helice;chassis;lipo;li-ion;18650;nimh;driver;l298;controleur;esc;ultrason;hc-sr04;suiveur;dupont;moteur;servo;recepteur","exclure_mots_cles":"foret;vis;peinture;cle;douille;perceuse;enceinte;haut parleur;lora;audio;jack;fpga;nema;cnc;industriel;graisse;gonfleur","exemple":"Voiture RC -> moteur / driver / batterie / capteur ultrason"},
 {"categorie":"rf","attributs_similaire":"rftech","complement_categories":"carte;alimentation;connectique","complement_categories_toutes":"","complement_mots_cles":"arduino;esp32;raspberry;antenne;dupont;module;carte sim;adaptateur","exclure_mots_cles":"foret;vis;haut parleur;enceinte;fpga;altera;peinture","exemple":"NRF24/LoRa/433 -> carte MCU / antenne / dupont"},
 {"categorie":"electronique","attributs_similaire":"module_fn","complement_categories":"carte;connectique;composant;alimentation","complement_categories_toutes":"","complement_mots_cles":"arduino;raspberry;esp32;nano;uno;dupont;breadboard;resistance;potentiometre;support;afficheur;ecran;lcd;oled;capteur","exclure_mots_cles":"foret;perceuse;vis;cle;douille;peinture;fpga;altera;cyclone;projecteur;clé à chocs","exemple":"Relais/LCD/Module -> Arduino / dupont / resistance / afficheur"},
 {"categorie":"composant","attributs_similaire":"component","complement_categories":"electronique;connectique;carte","complement_categories_toutes":"","complement_mots_cles":"breadboard;plaque essai;dupont;resistance;support;arduino;esp32;led;afficheur;potentiometre","exclure_mots_cles":"foret;perceuse;vis;peinture;fpga;altera;cyclone;projecteur;clé à chocs","exemple":"Resistance/diode -> breadboard / dupont / support DIP"},
 {"categorie":"connectique","attributs_similaire":"connector","complement_categories":"carte;electronique;composant","complement_categories_toutes":"","complement_mots_cles":"arduino;raspberry;esp32;module;breadboard;capteur;ecran;led;afficheur","exclure_mots_cles":"foret;vis;peinture;cle;douille;fpga;altera","exemple":"Cable dupont -> carte / module / breadboard"},
 {"categorie":"mesure","attributs_similaire":"","complement_categories":"composant;connectique","complement_categories_toutes":"","complement_mots_cles":"sonde;electrode;pince;cable;cordon;fusible;pile 9v;crocodile","exclure_mots_cles":"foret;vis;peinture;arduino;esp32;fpga","exemple":"Multimetre -> sondes / cordons / pinces"},
 {"categorie":"soudure","attributs_similaire":"","complement_categories":"soudure;composant;connectique","complement_categories_toutes":"","complement_mots_cles":"etain;flux;panne;tresse;pompe a dessouder;troisieme main;decapant","exclure_mots_cles":"foret;perceuse;vis;arduino;esp32;fpga;fer a souder;poste soudure;masque","exemple":"Fer a souder -> etain / flux / panne (consommables)"},
 {"categorie":"solaire","attributs_similaire":"voltage_bucket","complement_categories":"batterie;alimentation;chargeur","complement_categories_toutes":"","complement_mots_cles":"batterie;controleur;mppt;regulateur;18650;lipo;onduleur","exclure_mots_cles":"foret;vis;peinture;fpga;arduino","exemple":"Panneau solaire -> regulateur MPPT / batterie"},
 {"categorie":"audio","attributs_similaire":"","complement_categories":"connectique;alimentation;chargeur","complement_categories_toutes":"","complement_mots_cles":"jack;aux;cable;chargeur;alimentation;support;amplificateur;haut parleur;micro;rca","exclure_mots_cles":"foret;vis;esp32;arduino;relais;fpga;capteur;ruban","exemple":"Enceinte BT -> cable jack / ampli / alim"},
 {"categorie":"outillage","attributs_similaire":"","complement_categories":"","complement_categories_toutes":"","complement_mots_cles":"","exclure_mots_cles":"","exemple":"Outil -> aucun complement IoT"},
 {"categorie":"mecanique","attributs_similaire":"","complement_categories":"","complement_categories_toutes":"","complement_mots_cles":"","exclure_mots_cles":"","exemple":"Boitier/vis -> aucun complement direct (sert aux cartes)"},
 {"categorie":"electrique","attributs_similaire":"","complement_categories":"","complement_categories_toutes":"","complement_mots_cles":"","exclure_mots_cles":"","exemple":"Disjoncteur/220V -> aucun complement IoT"},
]
if REGEN_RULES or not RULES_CSV.exists():
    pd.DataFrame(_DEFAULT_RULES).to_csv(RULES_CSV, index=False)
    print(f"Regles ecrites dans {RULES_CSV}  (REGEN_RULES={REGEN_RULES})")

_rules = pd.read_csv(RULES_CSV).fillna("")
def _spl(s): return [x.strip() for x in str(s).split(";") if x.strip()]

TYPE_ATTRIBUTES     = {r["categorie"]: _spl(r["attributs_similaire"])          for _,r in _rules.iterrows() if _spl(r["attributs_similaire"])}
COMPLEMENTARITY_MAP = {r["categorie"]: _spl(r["complement_categories"])        for _,r in _rules.iterrows() if _spl(r["complement_categories"])}
COMPLEMENT_BROAD    = {r["categorie"]: _spl(r["complement_categories_toutes"]) for _,r in _rules.iterrows()}
COMPLEMENT_KEYWORDS = {r["categorie"]: [normalize_text(k) for k in _spl(r["complement_mots_cles"])] for _,r in _rules.iterrows()}
COMPLEMENT_EXCLUDE  = {r["categorie"]: [normalize_text(k) for k in _spl(r.get("exclure_mots_cles",""))] for _,r in _rules.iterrows()}

print(f"OK Regles chargees -- {len(_rules)} categories (avec blackliste)")
_rules[["categorie","complement_mots_cles","exclure_mots_cles"]].head(6)


## Etape 4 : Agregation des produits

In [ ]:
products = df.drop_duplicates(subset=["product_handle"]).copy()
inv = df.groupby("product_handle").agg({"available (not editable)":"sum","on hand (current)":"sum","on hand (new)":"sum"}).reset_index()
inv.columns = ["product_handle","available_total","on_hand_current","on_hand_new"]
products = products.merge(inv, on="product_handle", how="left")
products["available_total"] = products["available_total"].fillna(0).astype(int)
products["product_profile"] = products["product_title"].fillna("") + " categorie " + products["category"].fillna("")
products = products[["product_handle","product_title","category","sku","specs","attrs","is_tool","clean_title","product_profile","available_total","on_hand_current","on_hand_new"]].reset_index(drop=True)
print(f"OK {len(products)} produits uniques | En stock: {len(products[products['available_total']>0])}")
products[["product_title","category","attrs","available_total"]].head(5)

### Token Hugging Face (optionnel, securise)
Modele public -> token non requis. Sinon : secret Colab ou fichier `.env` (jamais en dur, jamais pousse sur git).

In [ ]:
import os
def _load_dotenv(path=".env"):
    p = Path(path)
    if not p.exists(): return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
_KEYS = ["HF_TOKEN","GEMINI_API_KEY","GROQ_API_KEY","ANTHROPIC_API_KEY","OPENAI_API_KEY"]
try:
    from google.colab import userdata
    for _k in _KEYS:
        try:
            _v = userdata.get(_k)
            if _v: os.environ[_k] = _v
        except Exception:
            pass
except Exception:
    _load_dotenv()
if os.getenv("HF_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.getenv("HF_TOKEN")
_found = [k for k in _KEYS if os.getenv(k)]
print("Cles LLM detectees :", _found if _found else "aucune (le moteur fonctionne sans LLM)")


## Etape 5 : Moteur de recommandation

In [ ]:
# ============================================================================
# Etape 5 -- MOTEUR DE RECOMMANDATION v2 (semantique, embeddings E5)
#   - Recherche produit ROBUSTE  : index/handle/SKU/titre exact -> sous-chaine -> semantique -> fuzzy
#   - SIMILAIRES   (sim=)    : MEME famille + meme attribut DEFINISSANT. Pour les categories SANS
#                              definissant (mecanique/connectique/mobilite...), on exige un LIEN DE
#                              FAMILLE reel (token-modele ou mot significatif partage) sinon RIEN.
#   - COMPLEMENTAIRES (compat=) : autre famille, compatibilite DURE ; tension seule insuffisante ;
#                              jamais un outil ni du bruit industriel ; diversite.
#   Repli automatique sur TF-IDF si sentence-transformers indisponible.
# ============================================================================
try:
    from rapidfuzz import process as _rf_process, fuzz as _rf_fuzz
    HAS_FUZZ = True
except Exception:
    HAS_FUZZ = False

# ---------------------------------------------------------------- engine
class SmartRecoEngine:
    EMB_MODEL    = "intfloat/multilingual-e5-base"
    RERANK_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
    STRONG = re.compile(r'^(?=.*[a-z])(?=.*\d)[a-z0-9]{3,}$')
    BLACKLIST = {"220v","380v","secteur","automate","simatic","plc","disjoncteur","contacteur",
                 "sectionneur","industrial","triphase","cn3791","variateur","profibus","fpga","altera","cyclone","cbb65","climatiseur"}
    # Attribut DEFINISSANT : doit coincider pour qu'un produit soit un VRAI substitut.
    DEFINING = {"batterie":"form_factor","carte":"board","composant":"component","capteur":"sensor",
                "electronique":"module_fn","moteur":"motor","led":"led_form","rf":"rftech","mobilite":"vehicle"}
    # Indices FORTS qu'un produit est un OUTIL / industriel (au-dela des marques connues) :
    # ces items polluent les recos quand ils sont mal categorises (foret en 'composant', etc.).
    TOOL_LIKE = re.compile(r'\b(foret|forets|meche|meches|ponceuse|souffleur|meuleuse|disqueuse|'
        r'tronconneuse|perceuse|visseuse|rabot|decapeur|compresseur|pistolet|peinture|burin|truelle|'
        r'riveteuse|agrafeuse|massette|marteau|tournevis|cutter|sangle|echelle|escabeau|brouette|cric|'
        r'palan|compression moteur|moteur essence|poste a souder|poste de soudure|\bmma\b|450kg|'
        r'\bscie\b|lame de scie|cisaille|nettoyeur|electrogene|aspirateur|taraud|filiere|mandrin|'
        r'extracteur|arrache|culasse|injecteur|vilebrequin|soupape|frein|etrier|rotule|cardan|'
        r'cle mixte|cle plate|cle a pipe|cle polygonale|cle a molette|cle dynamo|cle a ergot|'
        r'cle a choc|cle hexagonale|cle torx|cle allen|jeu de cle|pince multiprise|pince universelle|'
        r'pince coupante|pince a denuder|pince a sertir|pince a bec|coupe cable|coupe boulon|coffret|'
        r'pompe a graisse|graisse|pompe a huile|fer a souder|poste a souder|station de soudage|'
        r'burineur|cloueur|agrafeur|deboucheur|nettoyeur haute pression|karcher|pulverisateur|'
        r'tournevisseuse|riveteur|scie sabre|scie circulaire|ponceur|gonfleur|manometre)\b')
    # Bruit "complement" : items industriels / domotique batiment qui ne sont PAS des complements
    # IoT-dev (un ESP32 ne se complete pas avec un detecteur de fumee secteur ou un capteur inductif).
    COMP_NOISE = re.compile(r'\b(inductif|capacitif|plafonnier|micro ?onde|fin de course|'
        r'detecteur de fumee|detecteur de mouvement a micro|sirene|cctv|surveillance|telerupteur|'
        r'controle d acces|serrure|badge|portail|barriere|interphone|parlophone|prise intelligente|'
        r'disjoncteur|contacteur|sectionneur|variateur)\b')
    # Seuils. En mode TF-IDF on abaisse car les scores sont plus bas.
    def _floors(self):
        # Similaires classes par le cosinus E5 (compresse 0.75-0.90) -> seuils hauts.
        if self.is_sparse:
            return dict(SIM_ALT=0.18, SIM_NODEF=0.12, COMP=0.06, LOOKUP=0.30)
        sim = dict(SIM_ALT=0.86, SIM_NODEF=0.86)
        # Les COMPLEMENTAIRES n'utilisent JAMAIS le cross-encoder (il penalise une autre famille)
        # -> plancher sur le cosinus E5, quel que soit le mode.
        return dict(COMP=0.85, LOOKUP=0.82, **sim)

    def __init__(self, products, rule_maps, use_st=None, use_reranker=True, verbose=True):
        self.products = products.reset_index(drop=True)
        self.comp_map     = rule_maps["COMPLEMENTARITY_MAP"]
        self.comp_broad   = rule_maps["COMPLEMENT_BROAD"]
        self.comp_keywords= rule_maps["COMPLEMENT_KEYWORDS"]
        self.comp_exclude = rule_maps["COMPLEMENT_EXCLUDE"]
        self.type_attrs   = rule_maps["TYPE_ATTRIBUTES"]
        self.products["clean_profile"]=self.products["product_profile"].fillna("").apply(normalize_text)
        self.products["strong"]=self.products["clean_profile"].apply(lambda s:{w for w in s.split() if self.STRONG.match(w)})
        self.cat_to_idx={c:list(g.index) for c,g in self.products.groupby("category")}
        self._cat=self.products["category"].tolist(); self._attrs=self.products["attrs"].tolist()
        self._tool=self.products["is_tool"].tolist(); self._stock=self.products["available_total"].tolist()
        self._strong=self.products["strong"].tolist(); self._title=self.products["product_title"].tolist()
        self._title_clean=self.products["clean_title"].tolist(); self._specs=self.products["specs"].tolist()
        # titre BRUT (garde 'de'/'a'/'d'' et lettres seules) -> pour outils/bruit (ex: 'pompe a graisse')
        self._title_raw=[norm_raw(t) for t in self._title]
        self._handle=self.products["product_handle"].astype(str).str.lower().tolist()
        self._skul=self.products["sku"].astype(str).str.lower().tolist()
        use_st = HAS_ST if use_st is None else (use_st and HAS_ST)
        self.ce=None  # plus de cross-encoder : il produisait des scores trompeurs (0.00 sur un bon similaire)
        if use_st:
            if verbose: print("Chargement E5 (embeddings semantiques)...")
            self.emb=SentenceTransformer(self.EMB_MODEL)
            self.X=self._encode_cached(list(self.products["clean_profile"]), verbose)
            self.is_sparse=False; mode="semantique E5"
        else:
            self.vec=TfidfVectorizer(max_features=5000, ngram_range=(1,2))
            self.X=self.vec.fit_transform(self.products["clean_profile"]); self.is_sparse=True; mode="TF-IDF (repli)"
        self.F=self._floors()
        if verbose: print(f"OK Moteur pret -- {len(self.products)} produits -- mode {mode}")

    def _encode_cached(self, profiles, verbose):
        """Encode E5 avec cache disque (cle = modele + contenu) pour iterer vite en local."""
        import hashlib, os
        key = hashlib.md5((self.EMB_MODEL + "||" + "\n".join(profiles)).encode("utf-8")).hexdigest()[:16]
        path = f"_emb_{key}.npy"
        if os.path.exists(path):
            if verbose: print(f"  (embeddings charges du cache {path})")
            return np.load(path)
        X = self.emb.encode(["passage: "+p for p in profiles], normalize_embeddings=True,
                            batch_size=64, show_progress_bar=verbose).astype(np.float32)
        try: np.save(path, X)
        except Exception: pass
        return X

    # ---- similarites
    def _sims(self, i):
        if self.is_sparse: return (self.X @ self.X[i].T).toarray().ravel()
        return self.X @ self.X[i]
    def _sims_vec(self, qv):
        if self.is_sparse: return (self.X @ qv.T).toarray().ravel()
        return self.X @ qv
    def _pair_sim(self, a, b):
        if self.is_sparse: return float((self.X[a] @ self.X[b].T).toarray().ravel()[0])
        return float(self.X[a] @ self.X[b])
    def _diverse(self, idxs, k, thr):
        """Garde au plus k indices en evitant les quasi-doublons (cosinus > thr)."""
        picked=[]
        for j in idxs:
            if len(picked)>=k: break
            if all(self._pair_sim(j,p)<=thr for p in picked): picked.append(j)
        return picked
    def _efftool(self, j):
        """Outil EFFECTIF = marque-outil connue OU titre de type outil/industriel (sur titre BRUT)."""
        return bool(self._tool[j]) or bool(self.TOOL_LIKE.search(self._title_raw[j]))
    # mots trop generiques pour constituer un "lien de famille" entre deux produits
    GENERIC = {"cable","fil","fils","male","femelle","female","module","kit","pour","avec","vers","set",
        "jeu","jeux","lot","pcs","piece","pieces","broche","broches","pin","pins","noir","blanc","blanche",
        "rouge","bleu","bleue","vert","verte","jaune","gris","rose","digital","numerique","mini","micro",
        "pro","plus","haute","qualite","sans","type","modele","serie","version","couleur","metre","metres",
        "watt","volt","ampere","original","universel","universelle","adaptateur","connecteur",
        "plastique","acier","metal","metallique","inox","aluminium","laiton","nylon","cuivre","silicone",
        "etanche","flexible","longueur","largeur","blanc","noire","petit","grand","nouveau","nouvelle"}
    def _salient(self, j):
        """Tokens 'modele' (lettre+chiffre, ex: esp32, 18650, l298, sg90) -- forte preuve de famille."""
        return {w for w in self._title_raw[j].split()
                if self.STRONG.match(w) and not re.fullmatch(r'\d+(v|mah|ah|mm|cm|w|a|ma|k)?', w)}
    def _sig_words(self, j):
        return {w for w in self._title_clean[j].split() if len(w)>=4 and w not in self.GENERIC}
    def _family_overlap(self, i, j):
        """Vrai lien de famille : un token-modele partage OU un mot significatif (non generique) partage."""
        if self._salient(i) & self._salient(j): return True
        return bool(self._sig_words(i) & self._sig_words(j))
    @staticmethod
    def _kw_hit(tc, kws):
        """Match de mot-cle par MOT ENTIER (evite 'resistance' dans 'photoresistance')."""
        for k in kws:
            if re.search(r'(?<![a-z0-9])'+re.escape(k)+r'(?![a-z0-9])', tc): return True
        return False
    def _qvec(self, text):
        if self.is_sparse: return self.vec.transform([normalize_text(text)])
        return self.emb.encode(["query: "+normalize_text(text)], normalize_embeddings=True)[0].astype(np.float32)
    def _rerank(self, query_title, idxs):
        """Retourne {idx: score[0,1]} via cross-encoder ; sinon {} (repli sur semantique)."""
        if self.ce is None or not idxs: return {}
        pairs=[(query_title, self._title[j]) for j in idxs]
        raw=np.asarray(self.ce.predict(pairs, show_progress_bar=False), dtype=np.float64)
        sc=1.0/(1.0+np.exp(-raw))
        return {j:float(s) for j,s in zip(idxs, sc)}

    # ---- recherche produit ROBUSTE
    def get_product_index(self, q, return_conf=False):
        if isinstance(q,(int,np.integer)):
            ok = 0<=int(q)<len(self.products); return (int(q) if ok else None) if not return_conf else ((int(q),1.0) if ok else (None,0.0))
        qs=str(q).strip(); ql=qs.lower()
        if ql in self._handle:  i=self._handle.index(ql); return (i,1.0) if return_conf else i
        if ql in self._skul and ql.strip():  i=self._skul.index(ql); return (i,1.0) if return_conf else i
        nq=normalize_text(qs)
        if nq in self._title_clean:  i=self._title_clean.index(nq); return (i,1.0) if return_conf else i
        m=self.products[self.products["product_title"].str.contains(re.escape(qs), case=False, na=False)]
        if len(m): i=int(m.index[0]); return (i,0.95) if return_conf else i
        # semantique
        S=self._sims_vec(self._qvec(qs)); j=int(np.argmax(S)); conf=float(S[j])
        if conf>=self.F["LOOKUP"]: return (j,conf) if return_conf else j
        # fuzzy lexical
        if HAS_FUZZ:
            best=_rf_process.extractOne(nq, self._title_clean, scorer=_rf_fuzz.token_set_ratio)
            if best and best[1]>=80:
                i=self._title_clean.index(best[0]); return (i,best[1]/100.0) if return_conf else i
        return (j,conf) if return_conf else j   # dernier recours : meilleur semantique

    def _rows(self, idx, S):
        cols=["product_title","category","attrs","specs","available_total"]
        if not idx: return pd.DataFrame(columns=cols+["score"])
        out=self.products.loc[idx, cols].copy(); out["score"]=[round(float(S.get(j,0.0)),3) for j in idx]
        return out.reset_index(drop=True)

    def recommend(self, query, n=3, in_stock_only=True):
        i=self.get_product_index(query)
        if i is None: return None
        cat=self._cat[i]; sa={k:v for k,v in self._attrs[i].items() if v is not None}; stool=self._tool[i]
        src_efftool=self._efftool(i)   # l'objet source est-il un outil (marque OU type) ?
        def ok(j): return (not in_stock_only) or self._stock[j]>0
        S=self._sims(i)

        # ===== SIMILAIRES =====
        # On compare outils-avec-outils : une perceuse -> perceuses ; une resistance -> JAMAIS un foret.
        pool=[j for j in self.cat_to_idx.get(cat,[]) if j!=i and ok(j) and self._efftool(j)==src_efftool]
        pool=sorted(pool, key=lambda j:S[j], reverse=True)[:60]
        rr=self._rerank(self._title[i], pool)
        sc={j: rr.get(j, float(S[j])) for j in pool}     # score unifie [0,1]-ish
        defining=self.DEFINING.get(cat); src_def=self._attrs[i].get(defining) if defining else None
        if src_def is not None:
            tier1=[j for j in pool if self._attrs[j].get(defining)==src_def]
            tier2=[j for j in pool if self._attrs[j].get(defining)!=src_def and sc[j]>=self.F["SIM_ALT"]]
            # carte / mobilite / batterie : substitut = MEME type strict (un Arduino n'est pas un ESP32,
            # une 18650 n'est pas une pile bouton, une voiture RC n'est pas un autre vehicule).
            if cat in ("carte","mobilite","batterie"):
                tier2=[]
        else:
            tier1=[j for j in pool if sc[j]>=self.F["SIM_NODEF"]]; tier2=[]
        pk,pv=primary_spec(self._specs[i]); ups=[]
        if pk:
            ups=[j for j in tier1 if primary_spec(self._specs[j])[0]==pk and primary_spec(self._specs[j])[1]>pv]
            ups=sorted(ups, key=lambda j:(primary_spec(self._specs[j])[1], sc[j]), reverse=True)
        rest=sorted([j for j in tier1 if j not in ups], key=lambda j:sc[j], reverse=True)
        tier2=sorted(tier2, key=lambda j:sc[j], reverse=True)
        cand=ups+rest+tier2
        # GARDE-FOU : pour les categories SANS attribut definissant fiable (mecanique, connectique,
        # alimentation... + mobilite dont 'vehicle' est trop grossier), le seul cosinus E5 attrape des
        # produits qui partagent juste des mots de surface (cable DVI pour du Dupont, ecrou pour un
        # boitier). On exige un vrai LIEN DE FAMILLE (token-modele ou mot significatif partage),
        # sinon on ne propose RIEN (mieux vaut vide que faux).
        if ((cat not in self.DEFINING) or cat=="mobilite") and (self._salient(i) or self._sig_words(i)):
            cand=[j for j in cand if self._family_overlap(i, j)]
        # diversite douce : on retire seulement les RE-LISTINGS quasi-identiques (garde les variantes)
        similars=self._diverse(cand, n, thr=0.985 if not self.is_sparse else 0.97)

        # ===== COMPLEMENTAIRES =====
        # On calcule les complements des que la categorie a des regles (comp_cats non vide).
        # Les vraies categories d'outils (outillage/mecanique/electrique) ont des regles VIDES
        # -> naturellement aucun complement. Mais un objet de marque-outil tombe dans une
        # categorie IoT (ex: fer a souder TOTAL -> 'soudure') garde ses complements (etain/flux).
        comp=[]; comp_scores={}
        comp_cats=self.comp_map.get(cat,[])
        if comp_cats:
            broad=set(self.comp_broad.get(cat,[]))
            ckw=self.comp_keywords.get(cat,[]); excl=self.comp_exclude.get(cat,[])
            volt_ok=cat not in ("batterie","chargeur")
            raw=[]
            for ci,cc in enumerate(comp_cats):
                cc_broad=cc in broad
                for j in self.cat_to_idx.get(cc,[]):
                    # un complement n'est JAMAIS un outil, ni du bruit industriel / domotique batiment
                    if j==i or not ok(j) or self._efftool(j): continue
                    tj=self._title_clean[j]
                    if any(b in tj for b in self.BLACKLIST) or self.COMP_NOISE.search(self._title_raw[j]): continue
                    if excl and any(e in tj for e in excl): continue
                    ja=self._attrs[j]; hard=0.0; qual=False   # qual = a-t-il un VRAI signal de compatibilite ?
                    if sa.get("board") and ja.get("board")==sa["board"]: hard+=0.6; qual=True
                    if sa.get("connector") and ja.get("connector")==sa["connector"]: hard+=0.5; qual=True
                    if sa.get("form_factor") and sa["form_factor"] in tj: hard+=0.6; qual=True
                    if sa.get("chemistry") and ja.get("chemistry")==sa["chemistry"]: hard+=0.3; qual=True
                    if self._kw_hit(tj, ckw): hard+=0.5; qual=True
                    if cc_broad: hard+=0.3; qual=True        # categorie compagnon (ex: capteur pour une carte)
                    # La TENSION SEULE ne qualifie PAS (sinon tout produit 12V passe : thermostat, projecteur,
                    # kit acces...). Elle n'est qu'un BONUS de rang quand il y a deja un autre signal.
                    if volt_ok and sa.get("voltage_bucket") and ja.get("voltage_bucket")==sa["voltage_bucket"]: hard+=0.4
                    if not qual: continue                    # aucune compatibilite reelle -> rejete
                    raw.append((j, min(hard,1.0), ci))
            # Classement par COMPATIBILITE DURE (board/connecteur/mot-cle...), cosinus E5 en tie-break.
            # Le score AFFICHE est cette compatibilite (0-1) : eleve = vrai complement, bas = lien faible.
            order=sorted(raw, key=lambda c:(-c[1], -float(S[c[0]]), c[2]))
            comp_scores={j:h for j,h,_ in order}
            comp=self._diverse([j for j,_,_ in order], n, thr=0.94 if not self.is_sparse else 0.85)

        allsc={**{j:sc.get(j,float(S[j])) for j in similars}}
        if comp: allsc.update(comp_scores)
        return {"source_idx":i,"source_title":self._title[i],"source_cat":cat,"source_attrs":sa,
                "is_tool":bool(stool),"similars":self._rows(similars,allsc),"complements":self._rows(comp,allsc)}


_rule_maps = {"TYPE_ATTRIBUTES":TYPE_ATTRIBUTES, "COMPLEMENTARITY_MAP":COMPLEMENTARITY_MAP,
              "COMPLEMENT_BROAD":COMPLEMENT_BROAD, "COMPLEMENT_KEYWORDS":COMPLEMENT_KEYWORDS,
              "COMPLEMENT_EXCLUDE":COMPLEMENT_EXCLUDE}
engine = SmartRecoEngine(products, _rule_maps)


In [ ]:
# ============================================================================
# Etape 5bis -- CERVEAU LLM (expert humain) : parmi les candidats du moteur, il CHOISIT
#   les meilleurs similaires/complementaires et les EXPLIQUE (comme un vendeur expert).
#   Il ne pioche QUE dans le vrai stock -> aucune invention.
#   Multi-fournisseurs (auto-detecte la cle) : Gemini / Groq / Anthropic / OpenAI / Hugging Face.
#   Repli automatique sur le moteur si aucun LLM / limite de debit. USE_LLM=False pour desactiver.
# ============================================================================
USE_LLM = True

import os, re, json

def llm_provider():
    if os.environ.get("GEMINI_API_KEY"):    return "gemini"
    if os.environ.get("GROQ_API_KEY"):      return "groq"
    if os.environ.get("ANTHROPIC_API_KEY"): return "anthropic"
    if os.environ.get("OPENAI_API_KEY"):    return "openai"
    if os.environ.get("HF_TOKEN"):          return "hf"
    return None

HF_MODEL = "meta-llama/Llama-3.3-70B-Instruct"

def llm_chat(prompt, max_tokens=700, temperature=0.2, retries=1):
    import time
    last = None
    for attempt in range(retries + 1):
        try:
            return _llm_call(prompt, max_tokens, temperature)
        except Exception as e:                  # ex: limite de debit HF (429) -> petit backoff puis retry
            last = e
            if attempt < retries: time.sleep(2.5)
    raise last

def _llm_call(prompt, max_tokens=700, temperature=0.2):
    p = llm_provider()
    if p == "gemini":   # endpoint OpenAI-compatible de Google -> un seul SDK (openai) pour 3 fournisseurs
        from openai import OpenAI
        c = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
                   api_key=os.environ["GEMINI_API_KEY"])
        r = c.chat.completions.create(model="gemini-1.5-flash",
              messages=[{"role":"user","content":prompt}], max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    if p == "groq":
        from openai import OpenAI
        c = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.environ["GROQ_API_KEY"])
        r = c.chat.completions.create(model="llama-3.3-70b-versatile",
              messages=[{"role":"user","content":prompt}], max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    if p == "anthropic":
        import anthropic
        c = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        r = c.messages.create(model="claude-3-5-haiku-latest", max_tokens=max_tokens,
              messages=[{"role":"user","content":prompt}])
        return r.content[0].text
    if p == "openai":
        from openai import OpenAI
        c = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        r = c.chat.completions.create(model="gpt-4o-mini",
              messages=[{"role":"user","content":prompt}], max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    if p == "hf":
        from huggingface_hub import InferenceClient
        c = InferenceClient(model=HF_MODEL, token=os.environ["HF_TOKEN"])
        r = c.chat_completion(messages=[{"role":"user","content":prompt}],
              max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    raise RuntimeError("Aucun LLM configure")

def _parse_json(text):
    m = re.search(r"\[.*\]", text, re.DOTALL)
    if not m: return []
    try: return json.loads(m.group(0))
    except Exception:
        try: return json.loads(m.group(0).replace("'", '"'))
        except Exception: return []

_KINDDEF = {
 "similaires": "Un SIMILAIRE REMPLACE le produit (meme type/usage) : le client hesiterait entre les deux.",
 "complementaires": "Un COMPLEMENTAIRE s'achete EN PLUS pour faire fonctionner ou ameliorer le produit (autre type).",
}

def llm_pick(source_title, source_cat, df, kind, n=3):
    """df = DataFrame de candidats (product_title, category...). Retourne df reduit + colonne 'raison'."""
    if df is None or len(df) == 0: return df, "vide"
    titres = list(df["product_title"])
    lignes = "\n".join(f"{i+1}. {t}" for i, t in enumerate(titres))
    prompt = (
      f"Tu es un vendeur EXPERT en electronique/IoT. Un client regarde ce produit :\n"
      f'  "{source_title}"  (famille: {source_cat})\n\n'
      f"Voici des produits du MEME magasin, deja pre-selectionnes. "
      f"Choisis les {n} MEILLEURS {kind.upper()} pour ce client.\n{lignes}\n\n"
      f"Definition : {_KINDDEF[kind]}\n"
      f"Regles STRICTES :\n"
      f"- Choisis UNIQUEMENT dans la liste ci-dessus, par numero. N'invente RIEN.\n"
      f"- Classe du plus pertinent au moins pertinent.\n"
      f"- Si moins de {n} sont vraiment pertinents, en proposer MOINS (mieux vaut peu et juste).\n"
      f'- Donne une raison COURTE et humaine (ton de vendeur) pour chacun.\n'
      f'Reponds en JSON STRICT, rien d\'autre : [{{"n": <numero>, "raison": "<raison>"}}]'
    )
    try:
        data = _parse_json(llm_chat(prompt))
        rows, seen = [], set()
        for d in data:
            k = int(d.get("n", 0)) - 1
            if 0 <= k < len(titres) and k not in seen:
                seen.add(k); rows.append((k, str(d.get("raison", "")).strip()))
            if len(rows) >= n: break
        if not rows: raise ValueError("aucun choix LLM valide")
        out = df.iloc[[k for k, _ in rows]].copy()
        out["raison"] = [r for _, r in rows]
        return out.reset_index(drop=True), "llm:" + (llm_provider() or "?")
    except Exception as e:
        out = df.head(n).copy(); out["raison"] = ""        # REPLI : classement du moteur
        return out.reset_index(drop=True), f"repli-moteur ({type(e).__name__})"

def recommend_expert(engine, query, n=3, pool=12, in_stock_only=True):
    res = engine.recommend(query, n=pool, in_stock_only=in_stock_only)
    if res is None: return None
    sim, sm = llm_pick(res["source_title"], res["source_cat"], res["similars"], "similaires", n)
    comp, cm = llm_pick(res["source_title"], res["source_cat"], res["complements"], "complementaires", n)
    res["similars"], res["complements"] = sim, comp
    res["mode_sim"], res["mode_comp"] = sm, cm
    return res


if USE_LLM and llm_provider():
    print(f"Cerveau LLM actif : {llm_provider()}")
else:
    print("LLM non configure -> recommandations par le moteur embeddings seul (toujours fonctionnel).")


## Etape 6 : Tests

In [ ]:
def _attrs_str(a):
    a = {k:v for k,v in (a or {}).items() if v is not None}
    return ", ".join(f"{k}={v}" for k,v in a.items()) if a else "--"

def show_smart(query, n=3, in_stock_only=True):
    """SIMILAIRES + COMPLEMENTAIRES. Si un LLM est configure, il choisit + EXPLIQUE (ton humain)."""
    use_llm = USE_LLM and llm_provider()
    res = recommend_expert(engine, query, n=n, in_stock_only=in_stock_only) if use_llm \
          else engine.recommend(query, n=n, in_stock_only=in_stock_only)
    if res is None:
        print(f"X Produit introuvable : '{query}'"); return
    tag = f"  [cerveau: {res.get('mode_sim','moteur')}]" if use_llm else ""
    print("="*100)
    print(f"SELECTION : {res['source_title'][:80]}   [{res['source_cat']}]{tag}")
    print("="*100)
    for titre, dfb, sc in [("PRODUITS SIMILAIRES", res["similars"], "sim"),
                           ("PRODUITS COMPLEMENTAIRES", res["complements"], "compat")]:
        print(f"\n{titre} :")
        if len(dfb) == 0: print("   (aucun en stock)"); continue
        for _, r in dfb.iterrows():
            base = f"   - [{r['category']:<12}] {r['product_title'][:52]:<52}  {sc}={r['score']:.2f}"
            raison = r.get("raison", "") if "raison" in dfb.columns else ""
            print(base + (f"\n        -> {raison}" if raison else ""))
    print()


## Cellule Test Global

In [ ]:
# == Demo : 6 produits varies. Si un LLM est configure, il choisit + EXPLIQUE (ligne ->).
# (HF gratuit limite les rafales ; pour enchainer beaucoup de produits, prends une cle Groq/Gemini gratuite.)
for _q in ["Arduino UNO", "ESP32", "Ruban LED RGB 5050", "Batterie 18650", "Capteur DHT11", "Module relais 4 canaux"]:
    show_smart(_q)


## Etape 7 : Regler le moteur via le CSV
Edite `regles_recommandation.csv` (attributs du meme type, complements, mots-cles) puis re-execute a partir de l'Etape 5.

In [ ]:
# Pour regler : edite regles_recommandation.csv puis re-execute a partir de l'Etape 5.
# Exemple de modif en direct :
COMPLEMENTARITY_MAP["mobilite"] = ["batterie","chargeur","moteur","carte"]
engine.comp_map = COMPLEMENTARITY_MAP
print("Carte de complementarite mise a jour")
show_smart("Drone")

## Etape 8 : Evaluation -- precision REELLE sur jeu de test etiquete
(remplace l'audit tautologique "0 violation" par une vraie mesure Precision@k)


In [ ]:
# ============================================================================
# Etape 8 -- EVALUATION HONNETE : precision REELLE sur un jeu de test ETIQUETE
#   _hit teste le MOT ENTIER (pas la sous-chaine) -> l'eval ne se valide pas elle-meme
#   sur un mot (ex: "lcd" ne compte pas via "Stylo 3D LCD").
# ============================================================================
GOLD = [
 # ---------------- batteries / piles ----------------
 dict(q="Batterie Lithium 18650 3.7V 2600mAh", cat="batterie",
      sim_ok=["18650"], comp_cats=["chargeur","composant","mecanique","connectique"],
      comp_good=["chargeur","18650","bms","support","porte pile","boitier"],
      comp_bad=["diode","triac","resistance","ruban","moteur","foret","disjoncteur"]),
 dict(q="PILE GPU AA ULTRA PLUS BP4", cat="batterie",
      sim_ok=["aa","lr6","pile"], comp_cats=["chargeur","composant","mecanique","connectique"],
      comp_good=["chargeur","support","porte pile","boitier"],
      comp_bad=["diode","triac","ruban led","moteur","foret"]),
 dict(q="PILE RECHARGEABLE D RL20 4500 MAH", cat="batterie",
      sim_ok=["rl20","pile","rechargeable"], comp_cats=["chargeur","composant","mecanique","connectique"],
      comp_good=["chargeur","support","boitier"], comp_bad=["triac","ruban","foret","disjoncteur"]),
 # ---------------- cartes / MCU ----------------
 dict(q="Carte de développement Arduino Nano Officiel", cat="carte",
      sim_ok=["arduino","nano","uno","mega","atmega","leonardo"],
      comp_cats=["capteur","connectique","alimentation","led","electronique","composant"],
      comp_good=["dupont","breadboard","capteur","ecran","lcd","oled","resistance","alimentation","relais"],
      comp_bad=["foret","perceuse","disjoncteur","fpga","altera","cle a choc","visseuse"]),
 dict(q="Kit de Démarrage ESP32 Développement IoT WiFi Bluetooth", cat="carte",
      sim_ok=["esp32","esp8266","wroom","nodemcu"],
      comp_cats=["capteur","connectique","alimentation","led","electronique","composant"],
      comp_good=["dupont","breadboard","capteur","ecran","oled","lcd","relais","alimentation"],
      comp_bad=["foret","perceuse","disjoncteur","fpga","altera","visseuse"]),
 dict(q="Raspberry Pi Zero 2 W avec connecteurs", cat="carte",
      sim_ok=["raspberry","pi","rpi"],
      comp_cats=["capteur","connectique","alimentation","led","electronique","composant"],
      comp_good=["micro sd","carte memoire","dissipateur","alimentation","boitier","dupont","ecran"],
      comp_bad=["foret","perceuse","disjoncteur","fpga","altera"]),
 # ---------------- capteurs ----------------
 dict(q="Capteur de gaz SGP30 qualité de l air", cat="capteur",
      sim_ok=["capteur","gaz","sgp30","co2","mq","tvoc"],
      comp_cats=["carte","connectique","electronique"],
      comp_good=["arduino","esp32","raspberry","dupont","ecran","lcd","oled"],
      comp_bad=["foret","perceuse","fpga","altera","cle a choc"]),
 dict(q="Capteur de débit d eau YF-S401 débitmètre", cat="capteur",
      sim_ok=["capteur","debit","yf","debitmetre"],
      comp_cats=["carte","connectique","electronique"],
      comp_good=["arduino","esp32","dupont","ecran"], comp_bad=["foret","fpga","altera"]),
 # ---------------- LED ----------------
 dict(q="Ruban led RGB STRIP LIGHT 12V 5M", cat="led",
      sim_ok=["ruban","strip","bande","rgb"],
      comp_cats=["alimentation","electronique","connectique"],
      comp_good=["alimentation","controleur","dimmer","transformateur","amplificateur rgb"],
      comp_bad=["foret","vis","capteur","fpga","diode","projecteur"]),
 dict(q="Ruban LED COB 12V 8mm 5M 6500K", cat="led",
      sim_ok=["ruban","cob","bande","strip"],
      comp_cats=["alimentation","electronique","connectique"],
      comp_good=["alimentation","controleur","dimmer","transformateur"],
      comp_bad=["foret","vis","capteur","fpga","projecteur"]),
 # ---------------- electronique / modules ----------------
 dict(q="Module Relais 4 canaux 5V", cat="electronique",
      sim_ok=["relais","relay"],
      comp_cats=["carte","connectique","composant","alimentation"],
      comp_good=["arduino","esp32","raspberry","dupont","alimentation"],
      comp_bad=["foret","perceuse","fpga","altera","projecteur"]),
 dict(q="1602 MODULE AFFICHEUR LCD 2X16 AVEC INTERFACE I2C", cat="electronique",
      sim_ok=["lcd","oled","afficheur","ecran","1602","12864"],
      comp_cats=["carte","connectique","composant","alimentation"],
      comp_good=["arduino","esp32","dupont","resistance","potentiometre"],
      comp_bad=["foret","perceuse","fpga","altera"]),
 # ---------------- moteurs ----------------
 dict(q="Moteur pas à pas NEMA23 JK57HS41", cat="moteur",
      sim_ok=["moteur","nema","pas pas","stepper"],
      comp_cats=["carte","electronique","alimentation","batterie","mecanique"],
      comp_good=["driver","a4988","l298","controleur","arduino","esp32","alimentation"],
      comp_bad=["foret","perceuse","cle a choc","visseuse","meuleuse","fpga"]),
 dict(q="Mini Moteur 45000 RPM 7x20mm", cat="moteur",
      sim_ok=["moteur","motor","rpm"],
      comp_cats=["carte","electronique","alimentation","batterie","mecanique"],
      comp_good=["driver","l298","controleur","batterie","alimentation"],
      comp_bad=["foret","cle a choc","visseuse","meuleuse"]),
 # ---------------- composants ----------------
 dict(q="Diode 1N4007 1A 1000V", cat="composant",
      sim_ok=["diode","1n4007","1n4148","1n5408"],
      comp_cats=["electronique","connectique","carte"],
      comp_good=["breadboard","plaque essai","dupont","support","resistance"],
      comp_bad=["foret","perceuse","fpga","altera","projecteur","cle a choc"]),
 dict(q="Condensateur polyester 0.33µF 100V", cat="composant",
      sim_ok=["condensateur"],
      comp_cats=["electronique","connectique","carte"],
      comp_good=["breadboard","dupont","support","resistance"],
      comp_bad=["foret","fpga","altera","cle a choc"]),
 # ---------------- connectique ----------------
 dict(q="Cable Micro-USB 0.5m", cat="connectique",
      sim_ok=["cable","micro usb","usb"],
      comp_cats=["carte","electronique","composant"],
      comp_good=["arduino","esp32","raspberry","module","breadboard"],
      comp_bad=["foret","vis","fpga","altera"]),
 # ---------------- chargeur ----------------
 dict(q="Chargeur Batterie 18650 - Double", cat="chargeur",
      sim_ok=["chargeur","18650"], comp_cats=["batterie","connectique"],
      comp_good=["18650","batterie","pile","accu","cellule","li-ion","lipo"],
      comp_bad=["foret","vis","peinture","moteur"]),
 # ---------------- alimentation ----------------
 dict(q="Alimentation 12V 2A Adaptateur Secteur", cat="alimentation",
      sim_ok=["alimentation","12v","adaptateur"],
      comp_cats=["carte","connectique","led","electronique","mecanique"],
      comp_good=["arduino","esp32","raspberry","led","ruban","module","boitier","jack"],
      comp_bad=["foret","vis","cle","fpga","altera","triac"]),
 # ---------------- rf ----------------
 dict(q="Module LoRa 433 MHz émetteur récepteur", cat="rf",
      sim_ok=["lora","433","nrf24","rf"],
      comp_cats=["carte","alimentation","connectique"],
      comp_good=["arduino","esp32","raspberry","antenne","dupont","module"],
      comp_bad=["foret","vis","haut parleur","enceinte","fpga"]),
 # ---------------- audio ----------------
 dict(q="Haut Parleur Bluetooth TTD-8244", cat="audio",
      sim_ok=["haut parleur","bluetooth","enceinte","speaker"],
      comp_cats=["connectique","alimentation","chargeur"],
      comp_good=["jack","cable","chargeur","alimentation","aux","micro"],
      comp_bad=["foret","vis","esp32","arduino","relais","fpga","capteur","ruban"]),
 # ---------------- mesure ----------------
 dict(q="Multimètre digital true RMS 1000V", cat="mesure",
      sim_ok=["multimetre","testeur"],
      comp_cats=["composant","connectique"],
      comp_good=["sonde","cordon","pince","fusible","cable","crocodile"],
      comp_bad=["foret","vis","arduino","esp32","fpga"]),
 # ---------------- soudure ----------------
 dict(q="Fer À Souder 40 W TET1406", cat="soudure",
      sim_ok=["fer","souder","soudure"],
      comp_cats=["composant","connectique"],
      comp_good=["etain","flux","panne","tresse","support"],
      comp_bad=["foret","perceuse","arduino","esp32","fpga"]),
 # ---------------- solaire ----------------
 dict(q="Mini panneau solaire portable 12V 2W", cat="solaire",
      sim_ok=["solaire","panneau","photovolta"],
      comp_cats=["batterie","alimentation","chargeur"],
      comp_good=["batterie","controleur","mppt","regulateur","18650","onduleur"],
      comp_bad=["foret","vis","fpga","arduino"]),
 # ====== CAS DURS (categories faibles signalees par l'encadrant) -- tokens STRICTS ======
 # afficheur : un similaire doit etre un VRAI afficheur (pas un stylo 3D / jauge a ecran)
 dict(q="1602 MODULE AFFICHEUR LCD 2X16 I2C", cat="electronique",
      sim_ok=["lcd","oled","afficheur","1602","2004","12864","tft","nextion"],
      comp_cats=["carte","connectique","composant","alimentation"],
      comp_good=["arduino","esp32","dupont","resistance","potentiometre","nano","uno"],
      comp_bad=["stylo","jauge","fer a souder","foret","fpga"]),
 # voiture RC : similaire = autre vehicule pilotable (robot/2wd/rc), PAS un accessoire auto
 dict(q="Voiture Télécommandée 4WD haute vitesse", cat="mobilite",
      sim_ok=["robot","2wd","4wd","telecommand","radiocommand","drift"],
      comp_cats=["batterie","moteur","capteur","electronique","connectique","chargeur","rf"],
      comp_good=["moteur","driver","batterie","lipo","roue","servo"],
      comp_bad=["gonfleur","manometre","graisse","cle a choc","foret"]),
 # boitier : similaire = autre boitier/coque, PAS un DIAC/entretoise
 dict(q="Boîtier en plastique ABS pour Arduino UNO", cat="mecanique",
      sim_ok=["boitier","coque","case","abs","support"],
      comp_cats=[],   # mecanique : aucun complement attendu
      comp_good=[], comp_bad=["diac","triac","foret"]),
 # dupont : similaire = jumper/dupont, PAS un cable DVI/reseau
 dict(q="Cable Dupont jumper male male 40cm", cat="connectique",
      sim_ok=["dupont","jumper","cavalier"],
      comp_cats=["carte","electronique","composant"],
      comp_good=["arduino","esp32","breadboard","module","capteur"],
      comp_bad=["dvi","reseau","ethernet","foret"]),
 # helice drone : complement = moteur brushless/ESC, PAS un stepper industriel NEMA
 dict(q="Hélice fibre carbone 10x4.5 pour drone", cat="mobilite",
      sim_ok=["helice","pale","propeller"],
      comp_cats=["batterie","moteur","capteur","electronique","connectique","chargeur","rf"],
      comp_good=["brushless","esc","lipo","moteur"],
      comp_bad=["nema86","nema34","nema 86","cle a choc","foret"]),
]


def _hit(tc, toks):
    return any(re.search(r"(?<![a-z0-9])"+re.escape(t)+r"(?![a-z0-9])", tc) for t in toks)
def evaluer(verbose=True):
    import numpy as _np
    sim_p=[]; sim_f=[]; comp_p=[]; comp_g=[]; viol=0
    for g in GOLD:
        res = engine.recommend(g["q"], n=3, in_stock_only=False)
        if res is None:
            if verbose: print("X INTROUVABLE:", g["q"]); sim_f.append(0); continue
        sims=[normalize_text(t) for t in res["similars"]["product_title"]]
        comps=list(zip([normalize_text(t) for t in res["complements"]["product_title"]], res["complements"]["category"]))
        sc=sum(1 for s in sims if _hit(s, g["sim_ok"]))
        sim_p.append(sc/len(sims) if sims else 0.0); sim_f.append(1 if sc>0 else 0)
        cc=cb=cg=0
        for ct,cat in comps:
            bad=_hit(ct,g["comp_bad"]); cb+=bad
            if (cat in g["comp_cats"]) and not bad: cc+=1
            if _hit(ct,g["comp_good"]): cg=1
        comp_p.append(cc/len(comps) if comps else (1.0 if not g["comp_cats"] else 0.0))
        comp_g.append(cg); viol+=cb
    print(f"Similaires  : precision={_np.mean(sim_p):.0%} | trouve >=1 bon = {_np.mean(sim_f):.0%}")
    print(f"Complements : precision={_np.mean(comp_p):.0%} | a >=1 bon complement = {_np.mean(comp_g):.0%} | fautes graves = {viol}")
    print(f">>> SCORE GLOBAL (precision moyenne sim+comp) = {_np.mean(sim_p+comp_p):.1%}  sur {len(GOLD)} cas")
evaluer()


In [ ]:
# BONUS -- familles auto-decouvertes (clustering non supervise, preuve de self-discovery)
from sklearn.cluster import KMeans
K_FAMILIES = 20
km = KMeans(n_clusters=K_FAMILIES, random_state=42, n_init=4).fit(engine.X)
engine.products["auto_family"] = km.labels_
print(f"{K_FAMILIES} familles decouvertes automatiquement par le modele (sans regles ecrites a la main).")